In [ ]:
import pandas as pd
import sqlite3
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

In [ ]:
query_fatal = """
    SELECT r.pt 
    FROM reac_clean r
    JOIN outc_clean o ON r.primaryid = o.primaryid
    WHERE o.outc_cod IN ('DE', 'HO', 'LT')
"""
df_fatal = pd.read_sql_query(query_fatal, conn)
top_7_fatal = df_fatal['pt'].value_counts().head(7).index.tolist()

In [ ]:
df_all_reac = pd.read_sql_query("SELECT pt FROM reac_clean", conn)
common_counts = df_all_reac['pt'].value_counts()
top_8_common = common_counts[~common_counts.index.isin(top_7_fatal)].head(8).index.tolist()

In [ ]:
target_15_classes = top_7_fatal + top_8_common
print(f" Target Classes Identified: {len(target_15_classes)} classes")

In [ ]:
df_reac_full = pd.read_sql_query("SELECT primaryid, pt FROM reac_clean", conn)
df_target = df_reac_full[df_reac_full['pt'].isin(target_15_classes)].copy()

df_target = df_target.groupby('primaryid')['pt'].first().reset_index()
df_target.rename(columns={'pt': 'target_ae'}, inplace=True)

In [ ]:
query_demo = "SELECT primaryid, age, sex, wt, occr_country FROM demo_clean"
df_master_b = pd.read_sql_query(query_demo, conn)
df_master_b.drop_duplicates(subset=['primaryid'], inplace=True)

df_master_b['occr_country'] = df_master_b['occr_country'].astype(str).str.upper().str.strip().replace('NAN', 'UNK')

df_master_b = df_master_b.merge(df_target, on='primaryid', how='inner')

In [ ]:
df_drug = pd.read_sql_query("SELECT primaryid, role_cod, final_drug_name, rechal FROM drug_clean", conn)


df_drug['role_cod'] = df_drug['role_cod'].astype(str).str.upper().str.strip()
role_counts = pd.crosstab(df_drug['primaryid'], df_drug['role_cod']).reset_index()

expected_roles = ['PS', 'SS', 'C', 'I']
for role in expected_roles:
    if role not in role_counts.columns:
        role_counts[role] = 0

role_counts = role_counts[['primaryid', 'PS', 'SS', 'C', 'I']] # ترتيب ثابت
role_counts.rename(columns={
    'PS': 'num_primary_suspect', 
    'SS': 'num_secondary_suspect', 
    'C': 'num_concomitant', 
    'I': 'num_interacting'
}, inplace=True)


df_drug['positive_rechal'] = (df_drug['rechal'] == 'Y').astype(int)
rechal_feat = df_drug.groupby('primaryid')['positive_rechal'].max().reset_index(name='has_positive_rechallenge')


ps_drugs = df_drug[df_drug['role_cod'] == 'PS'].groupby('primaryid')['final_drug_name'].first().reset_index(name='primary_suspect_drug')

In [ ]:
df_indi = pd.read_sql_query("SELECT primaryid, indi_pt FROM indi_clean", conn)

df_indi['indi_pt'] = df_indi['indi_pt'].astype(str).str.upper().str.strip().replace('NAN', 'UNKNOWN_DIAGNOSIS')
primary_indication = df_indi.groupby('primaryid')['indi_pt'].first().reset_index(name='primary_diagnosis')


print("🔗 Assembling The Ultimate Matrix B...")
features = [role_counts, rechal_feat, ps_drugs, primary_indication]

for feat_df in features:
    df_master_b = df_master_b.merge(feat_df, on='primaryid', how='left')

df_master_b['primary_suspect_drug'] = df_master_b['primary_suspect_drug'].fillna('UNKNOWN_DRUG')
df_master_b['primary_diagnosis'] = df_master_b['primary_diagnosis'].fillna('UNKNOWN_DIAGNOSIS')

fillna_numeric = ['num_primary_suspect', 'num_secondary_suspect', 'num_concomitant', 'num_interacting', 'has_positive_rechallenge']
df_master_b[fillna_numeric] = df_master_b[fillna_numeric].fillna(0)

In [ ]:
print(f"Total Usable Patient Records: {len(df_master_b):,}")
print(f"Columns: {list(df_master_b.columns)}")

df_master_b.to_sql('matrix_b_ae', conn, if_exists='replace', index=False)
conn.close()

In [ ]:
# XGBoost (15 Target Classes)


import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Matrix B

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM matrix_b_ae", conn)
conn.close()

df.drop(columns=['primaryid'], inplace=True)

# 2. Encode the 15 Target Classes

le = LabelEncoder()
df['target_ae_encoded'] = le.fit_transform(df['target_ae'])

# Save mapping for later reference
target_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Target Classes Mapping: {target_mapping}\n")

X = df.drop(columns=['target_ae', 'target_ae_encoded'])
y = df['target_ae_encoded']


# 3. Native Categorical Setup for XGBoost

# XGBoost requires categorical features to be explicitly cast to 'category' dtype
categorical_cols = ['sex', 'occr_country', 'primary_suspect_drug', 'primary_diagnosis']
for col in categorical_cols:
    X[col] = X[col].astype('category')


# 4. Train/Test Split (Stratified to maintain class ratios)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training Set: {X_train.shape[0]:,} records")


# 5. Train XGBoost Model

xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',   # For multi-class probabilities
    num_class=15,                 # Explicitly declaring the number of classes
    max_depth=8,                  # Balanced depth
    learning_rate=0.1,
    n_estimators=300,
    random_state=42,
    tree_method='hist',           # Fast histogram-based execution
    enable_categorical=True       # Allow XGBoost to handle 'category' dtypes directly
)

xgb_model.fit(X_train, y_train)


# 6. Evaluation

y_pred = xgb_model.predict(X_test)

# Decode predictions for readable report
y_test_decoded = le.inverse_transform(y_test)
y_pred_decoded = le.inverse_transform(y_pred)

print(classification_report(y_test_decoded, y_pred_decoded))

# We focus on Macro F1-Score as it treats all classes equally, penalizing poor performance on minority classes
macro_f1 = f1_score(y_test_decoded, y_pred_decoded, average='macro')
weighted_f1 = f1_score(y_test_decoded, y_pred_decoded, average='weighted')

print(f" MACRO F1-SCORE: {macro_f1:.4f}")
print(f" WEIGHTED F1-SCORE: {weighted_f1:.4f}\n")

# 7. Feature Importance Plot

feature_importances = pd.Series(xgb_model.feature_importances_, index=X.columns)
top_features = feature_importances.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
plt.title('XGBoost Multi-Class Feature Importance', fontsize=16, pad=15)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Model 5.2: Multi-Class Baseline - LightGBM (15 Target Classes)

import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Matrix B
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM matrix_b_ae", conn)
conn.close()

df.drop(columns=['primaryid'], inplace=True)

# 2. Encode Target
le = LabelEncoder()
df['target_ae_encoded'] = le.fit_transform(df['target_ae'])

X = df.drop(columns=['target_ae', 'target_ae_encoded'])
y = df['target_ae_encoded']

# 3. Categorical Setup for LightGBM
categorical_cols = ['sex', 'occr_country', 'primary_suspect_drug', 'primary_diagnosis']
for col in categorical_cols:
    X[col] = X[col].astype('category')

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Train LightGBM Model

lgb_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=15,
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    random_state=42,
    n_jobs=-1 # Use all CPU cores
)

# LightGBM automatically detects 'category' dtypes in pandas
lgb_model.fit(X_train, y_train)

# 6. Evaluation

y_pred = lgb_model.predict(X_test)

y_test_decoded = le.inverse_transform(y_test)
y_pred_decoded = le.inverse_transform(y_pred)

print(classification_report(y_test_decoded, y_pred_decoded))

macro_f1 = f1_score(y_test_decoded, y_pred_decoded, average='macro')
weighted_f1 = f1_score(y_test_decoded, y_pred_decoded, average='weighted')

print(f" LIGHTGBM MACRO F1-SCORE: {macro_f1:.4f}")
print(f" LIGHTGBM WEIGHTED F1-SCORE: {weighted_f1:.4f}\n")

In [ ]:
# CatBoost (15 Target Classes)


import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns



# 1. Load Matrix B
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM matrix_b_ae", conn)
conn.close()

df.drop(columns=['primaryid'], inplace=True)

# 2. Encode Target
le = LabelEncoder()
df['target_ae_encoded'] = le.fit_transform(df['target_ae'])

X = df.drop(columns=['target_ae', 'target_ae_encoded'])
y = df['target_ae_encoded']

# 3. Define Categorical Features for CatBoost
# CatBoost expects categorical features as strings/objects, so we convert them
categorical_cols = ['sex', 'occr_country', 'primary_suspect_drug', 'primary_diagnosis']
for col in categorical_cols:
    X[col] = X[col].astype(str)

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Identify the index of categorical columns for CatBoost
cat_features_indices = [X.columns.get_loc(col) for col in categorical_cols]

# 5. Train CatBoost Model

cat_model = CatBoostClassifier(
    loss_function='MultiClass',
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_seed=42,
    task_type="CPU", # Change to "GPU" if you have a compatible NVIDIA GPU
    verbose=50       # Print progress every 50 iterations
)

cat_model.fit(
    X_train, y_train,
    cat_features=cat_features_indices,
    eval_set=(X_test, y_test),
    early_stopping_rounds=20
)

# 6. Evaluation

y_pred = cat_model.predict(X_test)

y_test_decoded = le.inverse_transform(y_test)
y_pred_decoded = le.inverse_transform(y_pred.flatten()) # Flatten because CatBoost outputs 2D array

print(classification_report(y_test_decoded, y_pred_decoded))

macro_f1 = f1_score(y_test_decoded, y_pred_decoded, average='macro')
weighted_f1 = f1_score(y_test_decoded, y_pred_decoded, average='weighted')

print(f" CATBOOST MACRO F1-SCORE: {macro_f1:.4f}")
print(f" CATBOOST WEIGHTED F1-SCORE: {weighted_f1:.4f}\n")

In [ ]:
# ==============================================================================
# Model 5.4: The Ultimate Ensemble (LightGBM + XGBoost) for Multi-Class
# ==============================================================================

import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import VotingClassifier
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

print("🚀 Initiating The Ultimate Ensemble (XGBoost + LightGBM)...\n")

# ---------------------------------------------------------
# 1. Load Matrix B & Encode
# ---------------------------------------------------------
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM matrix_b_ae", conn)
conn.close()

df.drop(columns=['primaryid'], inplace=True)

le = LabelEncoder()
df['target_ae_encoded'] = le.fit_transform(df['target_ae'])

X = df.drop(columns=['target_ae', 'target_ae_encoded'])
y = df['target_ae_encoded']

# ---------------------------------------------------------
# 2. Native Categorical Setup
# ---------------------------------------------------------
categorical_cols = ['sex', 'occr_country', 'primary_suspect_drug', 'primary_diagnosis']
for col in categorical_cols:
    X[col] = X[col].astype('category')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 3. Define the Base Models
# ---------------------------------------------------------
print("⚙️ Initializing Base Models...")
xgb_clf = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=15,
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    random_state=42,
    tree_method='hist',
    enable_categorical=True
)

lgb_clf = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=15,
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    verbose=-1 # Suppress LightGBM warnings
)

# ---------------------------------------------------------
# 4. Build and Train the Ensemble
# ---------------------------------------------------------
print("🧠 Training the Voting Classifier (Soft Voting)...")
# 'soft' voting means it averages the predicted probabilities from both models
ensemble_model = VotingClassifier(
    estimators=[('xgb', xgb_clf), ('lgb', lgb_clf)],
    voting='soft'
)

ensemble_model.fit(X_train, y_train)

# ---------------------------------------------------------
# 5. Evaluation
# ---------------------------------------------------------
print("\n📊 --- Ultimate Ensemble Classification Report ---")
y_pred = ensemble_model.predict(X_test)

y_test_decoded = le.inverse_transform(y_test)
y_pred_decoded = le.inverse_transform(y_pred)

print(classification_report(y_test_decoded, y_pred_decoded))

macro_f1 = f1_score(y_test_decoded, y_pred_decoded, average='macro')
weighted_f1 = f1_score(y_test_decoded, y_pred_decoded, average='weighted')

print(f"🌟 ENSEMBLE MACRO F1-SCORE: {macro_f1:.4f}")
print(f"🌟 ENSEMBLE WEIGHTED F1-SCORE: {weighted_f1:.4f}\n")

In [ ]:
import pandas as pd
import sqlite3
import numpy as np

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

# 1. Load Matrix B (V1) as our foundation

print("1. Loading Foundation Matrix B (V1)...")
df_master_b = pd.read_sql_query("SELECT * FROM matrix_b_ae", conn)


# 2. Extract Route of Administration (Primary Suspect)

print("2. Extracting Administration Route...")
query_route = """
    SELECT primaryid, route 
    FROM drug_clean 
    WHERE role_cod = 'PS'
"""
df_route = pd.read_sql_query(query_route, conn)


ps_route = df_route.groupby('primaryid')['route'].first().reset_index()
ps_route.rename(columns={'route': 'ps_route'}, inplace=True)
ps_route['ps_route'] = ps_route['ps_route'].astype(str).str.upper().str.strip().replace('NAN', 'UNKNOWN_ROUTE')

# 3. Calculate Time-to-Onset (TTO)

print("3. Calculating Time-to-Onset (TTO)...")

df_event = pd.read_sql_query("SELECT primaryid, event_dt FROM demo_clean", conn)

# Get Therapy Start Date for Primary Suspect Drug from THER (or DRUG depending on your schema)
# Note: FAERS usually stores start_dt in the THER table. We'll join with DRUG to ensure it's the PS drug.
try:
    query_dates = """
        SELECT t.primaryid, t.dsg_tbl_num, t.start_dt 
        FROM ther_clean t
        JOIN drug_clean d ON t.primaryid = d.primaryid AND t.dsg_tbl_num = d.drug_seq
        WHERE d.role_cod = 'PS'
    """
    df_start = pd.read_sql_query(query_dates, conn)
    df_start = df_start.groupby('primaryid')['start_dt'].first().reset_index()
except Exception as e:
    print(f" Warning: Could not execute complex THER join. Fetching generic start_dt if available. Error: {e}")
    # Fallback if schema differs
    df_start = pd.read_sql_query("SELECT primaryid, start_dt FROM ther_clean", conn)
    df_start = df_start.groupby('primaryid')['start_dt'].first().reset_index()

# Merge Dates
df_dates = df_event.merge(df_start, on='primaryid', how='left')

# Robust Datetime Parsing (FAERS dates are YYYYMMDD, but often messy like YYYY or YYYYMM)
def parse_faers_date(date_series):
    # Keep only exact 8-digit dates (YYYYMMDD) for accurate day calculation
    valid_dates = date_series.astype(str).str.extract(r'^(\d{8})$')[0]
    return pd.to_datetime(valid_dates, format='%Y%m%d', errors='coerce')

df_dates['event_dt_clean'] = parse_faers_date(df_dates['event_dt'])
df_dates['start_dt_clean'] = parse_faers_date(df_dates['start_dt'])

# Calculate Difference in Days
df_dates['tto_days'] = (df_dates['event_dt_clean'] - df_dates['start_dt_clean']).dt.days

# Clean up impossible values (e.g., negative TTO meaning symptom happened before drug)
df_dates.loc[df_dates['tto_days'] < 0, 'tto_days'] = np.nan 

tto_feature = df_dates[['primaryid', 'tto_days']]

# 4. Final Assembly (V2)

print("🔗 Assembling Matrix B (V2)...")
df_master_b_v2 = df_master_b.merge(ps_route, on='primaryid', how='left')
df_master_b_v2 = df_master_b_v2.merge(tto_feature, on='primaryid', how='left')

# Imputation for new columns
df_master_b_v2['ps_route'] = df_master_b_v2['ps_route'].fillna('UNKNOWN_ROUTE')

# For TTO, missing values are common. We'll fill with a special flag value (-1) 
# so the tree-based models can treat "Unknown TTO" as its own category/pattern.
df_master_b_v2['tto_days'] = df_master_b_v2['tto_days'].fillna(-1)

print(f"\n Matrix B V2 is complete!")
print(f"Total Usable Patient Records: {len(df_master_b_v2):,}")
print(f"Matrix V2 Columns: {list(df_master_b_v2.columns)}")

df_master_b_v2.to_sql('matrix_b_ae_v2', conn, if_exists='replace', index=False)
conn.close()

In [ ]:
#LightGBM 

import pandas as pd
import sqlite3
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns



# 1. Load Matrix B V2
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM matrix_b_ae_v2", conn)
conn.close()

df.drop(columns=['primaryid'], inplace=True)

# 2. Encode Target
le = LabelEncoder()
df['target_ae_encoded'] = le.fit_transform(df['target_ae'])

X = df.drop(columns=['target_ae', 'target_ae_encoded'])
y = df['target_ae_encoded']

# 3. Categorical Setup ( CRITICAL: Added 'ps_route' to categorical list)
categorical_cols = ['sex', 'occr_country', 'primary_suspect_drug', 'primary_diagnosis', 'ps_route']
for col in categorical_cols:
    X[col] = X[col].astype('category')

# Note: 'tto_days' is left as numeric naturally.

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Train LightGBM Model

lgb_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=15,
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(X_train, y_train)

# 6. Evaluation

y_pred = lgb_model.predict(X_test)

y_test_decoded = le.inverse_transform(y_test)
y_pred_decoded = le.inverse_transform(y_pred)

print(classification_report(y_test_decoded, y_pred_decoded))

macro_f1 = f1_score(y_test_decoded, y_pred_decoded, average='macro')
weighted_f1 = f1_score(y_test_decoded, y_pred_decoded, average='weighted')

print(f" V2 LIGHTGBM MACRO F1-SCORE: {macro_f1:.4f}")
print(f" V2 LIGHTGBM WEIGHTED F1-SCORE: {weighted_f1:.4f}\n")

# 7. Feature Importance Plot

feature_importances = pd.Series(lgb_model.feature_importances_, index=X.columns)
top_features = feature_importances.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_features.values, y=top_features.index, palette='mako')
plt.title('LightGBM V2 Feature Importance (Route & TTO Added)', fontsize=16, pad=15)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()